In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rahuljangir78/esg-and-financial-metrics-of-manufacturing-firms")

print("Path to dataset files:", path)

c:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\HP\.cache\kagglehub\datasets\rahuljangir78\esg-and-financial-metrics-of-manufacturing-firms\versions\1


In [3]:
df = pd.read_csv(r"C:\Users\HP\Desktop\ESG-Monitoring-System\Dataset\Manufacturing_ESG_Financial_Data.csv")

In [4]:
df.columns

Index(['Firm_ID', 'Year', 'Industry_Type', 'ESG_Score', 'E_Score', 'S_Score',
       'G_Score', 'ROA', 'ROE', 'Net_Profit_Margin', 'Revenue',
       'Operating_Cost', 'Firm_Size', 'Board_Independence',
       'Innovation_Spending'],
      dtype='str')

In [5]:
df["E_Compliance"] = df["E_Score"].apply(lambda x: "Violation" if x < 60 else "Compliant")
df["S_Compliance"] = df["S_Score"].apply(lambda x: "Violation" if x < 60 else "Compliant")
df["G_Compliance"] = df["G_Score"].apply(lambda x: "Violation" if x < 60 else "Compliant")


In [6]:
df["Overall_Compliance"] = df.apply(
    lambda row: "Non-Compliant"
    if "Violation" in [row["E_Compliance"], row["S_Compliance"], row["G_Compliance"]]
    else "Compliant",
    axis=1
)


In [7]:
df["compliance_risk_score"] = (
    (df["E_Compliance"] == "Violation").astype(int) +
    (df["S_Compliance"] == "Violation").astype(int) +
    (df["G_Compliance"] == "Violation").astype(int)
) * 33


In [8]:
df["performance_risk"] = (100 - df["ESG_Score"])


In [9]:
df["final_esg_risk_score"] = (
    df["performance_risk"] * 0.6 +
    df["compliance_risk_score"] * 0.4
)


In [10]:
def classify_alert(score):
    if score >= 70:
        return "Critical"
    elif score >= 40:
        return "Warning"
    else:
        return "Low Risk"

df["alert_level"] = df["final_esg_risk_score"].apply(classify_alert)


In [11]:
AGENT4_output = df[[
    "Firm_ID",
    "Year",
    "ESG_Score",
    "Overall_Compliance",
    "final_esg_risk_score",
    "alert_level"
]]


In [ ]:
AGENT4_output.head()

,Firm_ID,Year,ESG_Score,Overall_Compliance,final_esg_risk_score,alert_level
4995,MFG0401,2021,78.390000,Compliant,12.966,Low Risk
4996,MFG0233,2019,68.130000,Compliant,19.122,Low Risk
4997,MFG0158,2020,74.626667,Compliant,15.224,Low Risk
4998,MFG0437,2022,75.980000,Compliant,14.412,Low Risk
4999,MFG0269,2020,64.423333,Non-Compliant,34.546,Low Risk


In [13]:
AGENT4_output.to_csv("ESG4_output.csv", index=False)

In [14]:
AGENT4_json = AGENT4_output.to_json(orient="records")

In [15]:
AGENT4_json


'[{"Firm_ID":"MFG0103","Year":2023,"ESG_Score":64.7466666667,"Overall_Compliance":"Compliant","final_esg_risk_score":21.152,"alert_level":"Low Risk"},{"Firm_ID":"MFG0436","Year":2021,"ESG_Score":78.06,"Overall_Compliance":"Compliant","final_esg_risk_score":13.164,"alert_level":"Low Risk"},{"Firm_ID":"MFG0349","Year":2023,"ESG_Score":66.9366666667,"Overall_Compliance":"Non-Compliant","final_esg_risk_score":33.038,"alert_level":"Low Risk"},{"Firm_ID":"MFG0271","Year":2021,"ESG_Score":74.53,"Overall_Compliance":"Non-Compliant","final_esg_risk_score":28.482,"alert_level":"Low Risk"},{"Firm_ID":"MFG0107","Year":2019,"ESG_Score":68.15,"Overall_Compliance":"Non-Compliant","final_esg_risk_score":32.31,"alert_level":"Low Risk"},{"Firm_ID":"MFG0072","Year":2019,"ESG_Score":66.3566666667,"Overall_Compliance":"Compliant","final_esg_risk_score":20.186,"alert_level":"Low Risk"},{"Firm_ID":"MFG0189","Year":2019,"ESG_Score":66.69,"Overall_Compliance":"Compliant","final_esg_risk_score":19.986,"alert_le